# 2026Q3成分价格与披露版本补齐
使用当时已披露篮子，8月25日文件从8月26日使用；所有成员统一回看窗口，记录独立SEC价格锚点差异，不训练、不拟合调整系数。

In [1]:
from pathlib import Path
from datetime import datetime,timezone
from concurrent.futures import ThreadPoolExecutor,as_completed
import os,json,hashlib
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.market_update import _fetch_yahoo_chart_payload
from src.data.soxx_pit_holdings import build_pit_soxx_weight_panel
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/constituent_q3_extension_20261014';plan=json.loads((card/'plan.json').read_text());rawdir=root/plan['raw_directory'];rawdir.mkdir(parents=True,exist_ok=True)
for name,digest in plan['source_hashes'].items():assert hashlib.sha256((root/name).read_bytes()).hexdigest()==digest
old=pd.read_csv(root/plan['old_holdings']);new=pd.read_csv(rawdir/'new_holdings.csv');h=pd.concat([old,new],ignore_index=True).drop_duplicates(['accession','ticker'])
import exchange_calendars as xc
calendar=xc.get_calendar('XNYS',start=plan['quarter_start'],end=plan['as_of']).schedule.index
weights=build_pit_soxx_weight_panel(h,calendar=calendar,availability_lag_days=1)
assert (pd.to_datetime(weights.filing_date)<weights.datetime).all()
symbols=sorted(weights.instrument.unique());assert symbols==plan['symbols']
def fetch(symbol):
 path=rawdir/(symbol+'.json');rp=rawdir/(symbol+'_receipt.json')
 if not path.exists():
  payload=_fetch_yahoo_chart_payload(symbol,start=plan['price_start'],end=plan['as_of']);atomic_json(payload,path);atomic_json({'symbol':symbol,'retrieved_at':datetime.now(timezone.utc).isoformat(),'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'source':'Yahoo chart v8 via existing helper'},rp)
 payload=json.loads(path.read_text());receipt=json.loads(rp.read_text());assert hashlib.sha256(path.read_bytes()).hexdigest()==receipt['sha256'];return symbol,payload,receipt
payloads={};receipts={};errors={}
with ThreadPoolExecutor(max_workers=3) as pool:
 jobs={pool.submit(fetch,s):s for s in symbols}
 for future in as_completed(jobs):
  symbol=jobs[future]
  try:symbol,payload,receipt=future.result();payloads[symbol]=payload;receipts[symbol]=receipt
  except Exception as exc:errors[symbol]=str(exc)
atomic_json({'errors':errors,'receipts':receipts},rawdir/'fetch_manifest.json')
if errors:raise ValueError('needs_data: quote requests failed '+','.join(errors))
frames=[];counts=[];full_calendar=xc.get_calendar('XNYS',start=plan['price_start'],end=plan['as_of']).schedule.index
for symbol,payload in payloads.items():
 result=payload['chart']['result'][0];quote=result['indicators']['quote'][0];dates=pd.to_datetime(result['timestamp'],unit='s',utc=True).tz_convert('America/New_York').normalize().tz_localize(None)
 frame=pd.DataFrame({'datetime':dates,'instrument':symbol,**{n:quote[n] for n in ['open','high','low','close','volume']},'adj_close':result['indicators']['adjclose'][0]['adjclose']}).sort_values('datetime')
 assert pd.DatetimeIndex(frame.datetime).equals(full_calendar)
 assert np.isfinite(frame[['open','high','low','close','volume','adj_close']]).all().all() and frame[['open','high','low','close','adj_close']].gt(0).all().all()
 assert frame.high.ge(frame[['open','close','low']].max(axis=1)).all() and frame.low.le(frame[['open','close','high']].min(axis=1)).all() and frame.volume.ge(0).all()
 frames.append(frame);counts.append({'symbol':symbol,'rows':len(frame),'first':str(frame.datetime.min().date()),'last':str(frame.datetime.max().date())})
frame=pd.concat(frames,ignore_index=True);provider=root/plan['provider'];manifest=build_qlib_store_from_frame(frame,provider_uri=provider,freq='day',as_of=plan['as_of'],field_map={n:n for n in ['open','high','low','close','volume','adj_close']},source='Frozen Yahoo daily research snapshot with separate August25 SEC holdings receipt; retrospective adjustment, not live vintage')
qlib.init(provider_uri=str(provider),region='us');native=load_qlib_feature_frame(QlibProviderConfig(provider),instruments=symbols,fields=['$close','$adj_close'],start=plan['price_start'],end=plan['as_of']);prices=native['$adj_close'].unstack('instrument');ready=prices.rolling(61,min_periods=61).count().eq(61);coverage=[]
for day,part in weights.groupby('datetime'):
 valid=ready.loc[day,part.instrument].to_numpy();coverage.append({'date':day,'members':len(part),'covered_members':int(valid.sum()),'covered_weight_pct':float(part.loc[valid,'weight_pct'].sum()),'full_basket':bool(valid.all()),'accession':part.accession.iloc[0],'report_date':str(part.report_date.iloc[0].date()),'filing_date':str(part.filing_date.iloc[0].date())})
coverage=pd.DataFrame(coverage);assert len(coverage)==len(calendar) and coverage.full_basket.all()
# Independent end-June valuation anchors: mismatches are reported, never fitted away.
from src.data.soxx_pit_holdings import SOXX_NPORT_SECURITY_MAP,SOXX_NPORT_ISIN_MAP
anchors=[];tree=ET.fromstring((rawdir/'new_filing.xml').read_bytes());raw_close=native['$close'].unstack('instrument')
for node in tree.iter():
 if node.tag.split('}')[-1]!='invstOrSec':continue
 values={e.tag.split('}')[-1]:e.text for e in node};isin=next((e.attrib.get('value','') for e in node.iter() if e.tag.split('}')[-1]=='isin'),'');symbol=SOXX_NPORT_SECURITY_MAP.get(values.get('cusip',''),SOXX_NPORT_ISIN_MAP.get(isin))
 if symbol not in symbols or values.get('assetCat')!='EC':continue
 if values.get('units')!='NS' or values.get('curCd')!='USD':continue
 implied=float(values['valUSD'])/float(values['balance']);observed=float(raw_close.loc['2026-06-30',symbol]);anchors.append({'symbol':symbol,'cusip':values.get('cusip'),'sec_implied_price':implied,'native_close':observed,'absolute_difference':abs(implied-observed),'relative_difference':abs(implied-observed)/implied})
summary={'experiment':card.name,'decision':'Q3_reported_basket_prices_extended_no_model_change','new_fits':0,'new_backtests':0,'outcome_correlations':0,'objective_achieved':False,'symbols':len(symbols),'price_rows':len(frame),'q3_sessions':len(coverage),'full_basket_sessions':int(coverage.full_basket.sum()),'new_filing_date':'2026-08-25','new_report_date':'2026-06-30','earliest_new_report_use':'2026-08-26','independent_anchor_count':len(anchors),'anchor_rows_above_one_bp':sum(a['relative_difference']>.0001 for a in anchors),'maximum_anchor_relative_difference':max(a['relative_difference'] for a in anchors),'historical_first_receipt_certified':False,'actual_daily_index_membership_certified':False,'existing_sources_overwritten':False,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/prices','analysis/weights']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='quarter_price_extension',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0)
 rec.save_objects(artifact_path='inputs',plan=plan,holdings=h,sec_xml=(rawdir/'new_filing.xml').read_bytes(),sec_feed=(rawdir/'feed.xml').read_bytes(),payloads=payloads,receipts=receipts,provider_manifest=manifest)
 rec.save_objects(artifact_path='analysis',prices=native,weights=weights,coverage=coverage,anchors=pd.DataFrame(anchors),summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),full_basket_sessions=len(coverage))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(counts).to_csv(card/'price_coverage.csv',index=False);coverage.groupby(['report_date','filing_date','accession']).agg(sessions=('date','size'),complete=('full_basket','sum')).reset_index().to_csv(card/'holding_versions.csv',index=False);pd.DataFrame(anchors).to_csv(card/'price_anchor_checks.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
print({k:summary[k] for k in ['symbols','price_rows','q3_sessions','full_basket_sessions','independent_anchor_count','anchor_rows_above_one_bp','maximum_anchor_relative_difference']})


# 2026Q3已披露篮子数据补齐，不训练

本轮0拟合、0回测、0未来收益相关筛选。原成分价格只到7月2日，无法解释整个2026Q3；现独立取得30个相关成员2026-03-02至9月30日各148条日线，共4440条。每个标的完整NYSE日历、正价、OHLC及成交量检查通过；新Qlib provider和原始载荷独立保存，没有拼接不同复权版本的绝对价格水平，也不覆盖旧研究数据。

SEC Atom及原始XML确认新增报告：2026-06-30持仓于2026-08-25披露。季度前39个交易日使用此前已公开的2026-03-31报告（5月28日披露），后25日从8月26日起使用新报告。7月13日晚到旧报告没有覆盖更新报告期；采用已修复的报告期优先逻辑。同日提交一律不用于当天特征，不把6月底持仓提前用于7月。

季度64个交易日全部具备30成员价格及61日回看，64/64完整。原“2026Q3仅2/64日”的限制仍准确描述旧provider，不再代表所有可用来源。仍需承认N-PORT披露滞后：它是已披露篮子代理，不是官方每日指数成员和权重。

6月30日30个独立价格锚点核查：对SEC记录的USD/NS持仓，以市值除以股数，与Yahoo候选原收盘价的Qlib读回比较。没有超过1bp差异的记录；最大相对差4.92e-08，处于float32精度量级。外国证券由原XML ISIN映射；原始标识和报告均存Recorder。锚点只验证一个报告日，并不认证全区间历史首收或复权版本。

所有股票行情记录取回时间与载荷SHA256，原SEC XML、feed、持仓版本、价格与覆盖表已归档NAS。此数据是事后抓取的研究快照，historical_first_receipt_certified=false。没有新增预测、收益或生产策略变化。

决定：Q3_reported_basket_prices_extended_no_model_change。下一步可在这个预先选定的差区间核查内部上涨参与、新低扩散和下跌贡献集中度，区分ETF整体变化与少数权重股变化；仍不训练、不按未来收益选择窗口，不重开已关闭的四特征广度模型。

[逐标的价格覆盖](price_coverage.csv) · [季度持仓版本](holding_versions.csv) · [SEC价格锚点](price_anchor_checks.csv)

Recorder `50ec963e60254f5aa27aa3af6d4dfe36`。
